data source: [RCCGI](https://censusindia.gov.in/nada/index.php/catalog/44377)<br>
[List of Indian states by life expectancy at birth](https://en.wikipedia.org/wiki/List_of_Indian_states_by_life_expectancy_at_birth) / [Продолжительность жизни в штатах Индии](https://ru.wikipedia.org/wiki/Продолжительность_жизни_в_штатах_Индии)<br>
[MapChart](https://www.mapchart.net/india.html)

In [2]:
import pandas as pd
import math
import re
from collections import namedtuple

In [3]:
PERIOD = '2016-20'  # '2015-19' / '2016-20'

In [4]:
# load stats about longevity per year
df = pd.read_csv(f'data/SRS_{PERIOD}.csv', sep='\t', index_col='state', na_values='N.A.')
df.index.name = ''

# change order of columns to swap places rural and urban data
df = df.iloc[:, [0,1,2, 6,7,8, 3,4,5]]

# sort values
df.sort_values(by=['total', 'male'], ascending=False, inplace=True)
df = pd.concat([df.loc[['India']], df.drop('India')])

df.head(3).fillna('')

,total,male,female,urban_total,urban_male,urban_female,rural_total,rural_male,rural_female
,,,,,,,,,
India,70.0,68.6,71.4,73.2,71.9,74.5,68.6,67.2,70.1
Delhi,75.8,74.1,77.7,75.8,74.1,77.8,74.0,,76.6
Kerala,75.0,71.9,78.0,74.7,71.5,78.0,75.2,72.3,78.1


In [5]:
df.insert(loc=3, column='fΔm',  value=(df['female']-df['male']).round(1))
df.insert(loc=7, column='urban_fΔm',  value=(df['urban_female']-df['urban_male']).round(1))
df.insert(loc=11, column='rural_fΔm',  value=(df['rural_female']-df['rural_male']).round(1))
df.insert(loc=12, column='urbanΔrural',  value=(df['urban_total']-df['rural_total']).round(1))

df.fillna('')

,total,male,female,fΔm,urban_total,urban_male,urban_female,urban_fΔm,rural_total,rural_male,rural_female,rural_fΔm,urbanΔrural
,,,,,,,,,,,,,
India,70.0,68.6,71.4,2.8,73.2,71.9,74.5,2.6,68.6,67.2,70.1,2.9,4.6
Delhi,75.8,74.1,77.7,3.6,75.8,74.1,77.8,3.7,74.0,,76.6,,1.8
Kerala,75.0,71.9,78.0,6.1,74.7,71.5,78.0,6.5,75.2,72.3,78.1,5.8,-0.5
Jammu & Kashmir,74.3,72.6,76.3,3.7,78.1,76.0,80.5,4.5,72.7,71.1,74.6,3.5,5.4
Himachal Pradesh,73.5,70.3,77.5,7.2,77.1,74.7,81.0,6.3,73.2,69.9,77.2,7.3,3.9
Tamil Nadu,73.2,71.0,75.5,4.5,75.8,73.7,78.2,4.5,70.5,68.3,72.9,4.6,5.3
Maharashtra,72.9,71.6,74.3,2.7,74.6,73.4,76.1,2.7,71.6,70.2,73.0,2.8,3.0
Punjab,72.5,70.8,74.5,3.7,75.5,73.3,78.1,4.8,70.9,69.2,72.9,3.7,4.6
West Bengal,72.3,71.1,73.6,2.5,74.5,73.8,75.3,1.5,71.1,69.6,72.7,3.1,3.4


<br>
<br>

In [7]:
# Some inner statistics for interest:
def min_and_max(df, region_center=''):
    print(f"Number of values: {len(df)}")
    
    def find_3_largest(df):
        t_df = df.nlargest(3)
        t_df = t_df.reset_index()
        return t_df.apply(lambda v: f"{v.iloc[1]:.1f} – {v.iloc[0]}", axis=1)

    def find_3_smallest(df):
        t_df = df.nsmallest(3).iloc[::-1]
        t_df = t_df.reset_index()
        return t_df.apply(lambda v: f"{v.iloc[1]:.1f} – {v.iloc[0]}", axis=1)
    
    if region_center:
        ser_center = df.loc[region_center].map(lambda v: f" — {v} —")
        final_df = pd.concat([df.apply(find_3_largest), ser_center.to_frame().T, df.apply(find_3_smallest)])
        final_df.index = ['max', 'max_2', 'max_3', region_center, 'min_3', 'min_2', 'min']
    else:
        final_df = pd.concat([df.apply(find_3_largest), df.apply(find_3_smallest)])
        final_df.index = ['max', 'max_2', 'max_3', 'min_3', 'min_2', 'min']
    return final_df.style.set_properties(**{'text-align': 'left'})

min_and_max(df.loc[:, ['total', 'male', 'female', 'fΔm', 'urban_total', 'urban_male', 'urban_female', 'urban_fΔm', 'urbanΔrural']],
            region_center="India")

Number of values: 23


,total,male,female,fΔm,urban_total,urban_male,urban_female,urban_fΔm,urbanΔrural
max,75.8 – Delhi,74.1 – Delhi,78.0 – Kerala,7.2 – Himachal Pradesh,78.1 – Jammu & Kashmir,76.0 – Jammu & Kashmir,81.0 – Himachal Pradesh,6.5 – Kerala,7.4 – Assam
max_2,75.0 – Kerala,72.6 – Jammu & Kashmir,77.7 – Delhi,6.4 – Uttarakhand,77.1 – Himachal Pradesh,74.7 – Himachal Pradesh,80.5 – Jammu & Kashmir,6.3 – Himachal Pradesh,5.4 – Jammu & Kashmir
max_3,74.3 – Jammu & Kashmir,71.9 – Kerala,77.5 – Himachal Pradesh,6.1 – Kerala,75.8 – Delhi,74.1 – Delhi,78.2 – Tamil Nadu,5.6 – Haryana,5.3 – Tamil Nadu
India,— 70.0 —,— 68.6 —,— 71.4 —,— 2.8 —,— 73.2 —,— 71.9 —,— 74.5 —,— 2.6 —,— 4.6 —
min_3,67.4 – Madhya Pradesh,65.5 – Madhya Pradesh,68.6 – Assam,1.3 – Assam,70.8 – Madhya Pradesh,69.1 – Uttar Pradesh,71.3 – Bihar,0.4 – Telangana,1.8 – Delhi
min_2,66.0 – Uttar Pradesh,65.3 – Uttar Pradesh,66.8 – Chhattisgarh,-0.5 – Bihar,69.2 – Uttar Pradesh,68.8 – Uttarakhand,69.4 – Chhattisgarh,0.2 – Uttar Pradesh,0.7 – Uttarakhand
min,65.1 – Chhattisgarh,63.5 – Chhattisgarh,66.7 – Uttar Pradesh,-1.6 – Jharkhand,68.0 – Chhattisgarh,66.7 – Chhattisgarh,69.3 – Uttar Pradesh,-1.0 – Bihar,-0.5 – Kerala


<br>
<br>

In [9]:
# for state in sorted(df.index.to_list()):
#     print(f"    '{state}': {{'en': ('', ''), 'ru': ('', '')}},")

In [10]:
dd_replacement = {
    'India': {'en': ('India on average', ''), 'ru': ('Индия в среднем', '')},
    'Andhra Pradesh': {'en': ('Andhra Pradesh', 'Andhra Pradesh'), 'ru': ('А́ндхра-Праде́ш', 'Андхра-Прадеш')},
    'Assam': {'en': ('Assam', 'Assam'), 'ru': ('Асса́м', 'Ассам')},
    'Bihar': {'en': ('Bihar', 'Bihar'), 'ru': ('Биха́р', 'Бихар')},
    'Chhattisgarh': {'en': ('Chhattisgarh', 'Chhattisgarh'), 'ru': ('Чхаттисга́рх', 'Чхаттисгарх')},
    'Delhi': {'en': ('Delhi', 'Delhi'), 'ru': ('Де́ли', 'Дели')},
    'Gujarat': {'en': ('Gujarat', 'Gujarat'), 'ru': ('Гуджара́т', 'Гуджарат')},
    'Haryana': {'en': ('Haryana', 'Haryana'), 'ru': ('Харья́на', 'Харьяна')},
    'Himachal Pradesh': {'en': ('Himachal Pradesh', 'Himachal Pradesh'), 'ru': ('Химача́л-Праде́ш', 'Химачал-Прадеш')},
    'Jammu & Kashmir': {'en': ('Jammu and Kashmir', 'Jammu and Kashmir (union territory)'), 'ru': ('Джа́мму и Кашми́р', 'Джамму и Кашмир (союзная территория)')},
    'Jharkhand': {'en': ('Jharkhand', 'Jharkhand'), 'ru': ('Джаркха́нд', 'Джаркханд')},
    'Karnataka': {'en': ('Karnataka', 'Karnataka'), 'ru': ('Карна́така', 'Карнатака')},
    'Kerala': {'en': ('Kerala', 'Kerala'), 'ru': ('Ке́рала', 'Керала')},
    'Madhya Pradesh': {'en': ('Madhya Pradesh', 'Madhya Pradesh'), 'ru': ('Ма́дхья-Праде́ш', 'Мадхья-Прадеш')},
    'Maharashtra': {'en': ('Maharashtra', 'Maharashtra'), 'ru': ('Махара́штра', 'Махараштра')},
    'Odisha': {'en': ('Odisha', 'Odisha'), 'ru': ('Оди́ша', 'Одиша')},
    'Punjab': {'en': ('Punjab', 'Punjab, India'), 'ru': ('Пенджа́б', 'Пенджаб (Индия)')},
    'Rajasthan': {'en': ('Rajasthan', 'Rajasthan'), 'ru': ('Раджастха́н', 'Раджастхан')},
    'Tamil Nadu': {'en': ('Tamil Nadu', 'Tamil Nadu'), 'ru': ('Тамилна́д', 'Тамилнад')},
    'Telangana': {'en': ('Telangana', 'Telangana'), 'ru': ('Телингана', 'Телингана')},
    'Uttar Pradesh': {'en': ('Uttar Pradesh', 'Uttar Pradesh'), 'ru': ('У́ттар-Праде́ш', 'Уттар-Прадеш')},
    'Uttarakhand': {'en': ('Uttarakhand', 'Uttarakhand'), 'ru': ('Уттаракха́нд', 'Уттаракханд')},
    'West Bengal': {'en': ('West Bengal', ''), 'ru': ('Западная Бенга́лия', 'Западная Бенгалия')}
}

In [11]:
# create code for placing info in Wikipedia
def create_table(df, file_header, lang='ru'):

    def if_value(x, prec=1):
        return '—' if math.isnan(x) else \
               f"{x:0.{prec}f}"  if x>=0 else \
               f"−{-x:0.{prec}f}"                #"{x:0.{prec}f}".format(x, prec)

    with open('design/' + file_header, mode='r', encoding="utf-8") as fh:
        table_header = fh.read()

    st = ''
    for i in range(len(df)):
        ser = df.iloc[i]    
        if ser.name == 'India':
            st += '\n' + '|-class=static-row-header\n' + \
                  f'|style="text-align:left;"|\'\'\'{dd_replacement[ser.name][lang][0]}\'\'\' ' + \
                  f'|| style="text-align:center; background:#e0ffd8;"|\'\'\'{if_value(ser["total"])}\'\'\' ' + \
                  f'|| style="text-align:center;background:#eaf3ff;"|\'\'\'{if_value(ser["male"])}\'\'\' ' + \
                  f'|| style="text-align:center;background:#fee7f6;"|\'\'\'{if_value(ser["female"])}\'\'\' ' + \
                  f'|| style="padding-right:2ex;"|\'\'\'{if_value(ser["fΔm"])}\'\'\' ' + \
                  f'|| style="text-align:center;background:#e0ffd8;border-left-width:2px;"|\'\'\'{if_value(ser["urban_total"])}\'\'\' ' + \
                  f'|| style="text-align:center;background:#eaf3ff;"|\'\'\'{if_value(ser["urban_male"])}\'\'\' ' + \
                  f'|| style="text-align:center;background:#fee7f6;"|\'\'\'{if_value(ser["urban_female"])}\'\'\' ' + \
                  f'|| style="padding-right:2ex;"|\'\'\'{if_value(ser["urban_fΔm"])}\'\'\' ' + \
                  f'|| style="text-align:center;background:#e0ffd8;border-left-width:2px;"|\'\'\'{if_value(ser["rural_total"])}\'\'\' ' + \
                  f'|| style="text-align:center;background:#eaf3ff;"|\'\'\'{if_value(ser["rural_male"])}\'\'\' ' + \
                  f'|| style="text-align:center;background:#fee7f6;"|\'\'\'{if_value(ser["rural_female"])}\'\'\' ' + \
                  f'|| style="padding-right:2ex;"|\'\'\'{if_value(ser["rural_fΔm"])}\'\'\' ' + \
                  f'|| style="padding-right:2ex;border-left-width:2px;"|\'\'\'{if_value(ser["urbanΔrural"])}\'\'\''
        else:
            name_link = dd_replacement[ser.name][lang][1]
            name_visible = dd_replacement[ser.name][lang][0]
            name_inserted = name_link if name_link == name_visible else f"{name_link}|{name_visible}"
            st += '\n' + '|-\n' + \
                  f'|style="text-align:left;"|[[{name_inserted}]] ' + \
                  f'|| style="text-align:center; background:#e0ffd8;"|\'\'\'{if_value(ser["total"])}\'\'\' ' + \
                  f'|| style="text-align:center;background:#eaf3ff;"|{if_value(ser["male"])} ' + \
                  f'|| style="text-align:center;background:#fee7f6;"|{if_value(ser["female"])} ' + \
                  f'|| style="padding-right:2ex;"|{if_value(ser["fΔm"])} ' + \
                  f'|| style="text-align:center;background:#e0ffd8;border-left-width:2px;"|\'\'\'{if_value(ser["urban_total"])}\'\'\' ' + \
                  f'|| style="text-align:center;background:#eaf3ff;"|{if_value(ser["urban_male"])} ' + \
                  f'|| style="text-align:center;background:#fee7f6;"|{if_value(ser["urban_female"])} ' + \
                  f'|| style="padding-right:2ex;"|{if_value(ser["urban_fΔm"])} ' + \
                  f'|| style="text-align:center;background:#e0ffd8;border-left-width:2px;"|\'\'\'{if_value(ser["rural_total"])}\'\'\' ' + \
                  f'|| style="text-align:center;background:#eaf3ff;"|{if_value(ser["rural_male"])} ' + \
                  f'|| style="text-align:center;background:#fee7f6;"|{if_value(ser["rural_female"])} ' + \
                  f'|| style="padding-right:2ex;"|{if_value(ser["rural_fΔm"])} ' + \
                  f'|| style="padding-right:2ex;border-left-width:2px;"|{if_value(ser["urbanΔrural"])}'

    if lang == 'ru':
        st = re.sub('(?<=\\d)\\.(?=\\d)', ',', st)  # replace . to comma, if this . is between two digits
        st = st.replace('padding-right:1,5ex;', 'padding-right:1.5ex;')

    st = table_header + st + '\n|}'
    
    # gray color for missing values
    st = st.replace(';"|—', ';color:silver;"|—')

    return st


table_code = create_table(df, file_header='SRS_header_ru.txt', lang='ru')

# write the code to file
with open(f'output/Table code for Indian states by SRS_{PERIOD} -ru.txt', 'w', encoding='utf-8') as fh:
    fh.write(table_code)

In [12]:
table_code = create_table(df, file_header='SRS_header_en.txt', lang='en')

# write the code to file
with open(f'output/Table code for Indian states by SRS_{PERIOD} -en.txt', 'w', encoding='utf-8') as fh:
    fh.write(table_code)

<br>
<br>
<br>
<hr>

<h3>Map creation</h3>

In [14]:
CountryGroup = namedtuple('CountryGroup', ['group_label', 'color', 'countries'])

In [15]:
# for state in sorted(df.index.to_list()):
#     print(f"'{state}'", end=', ')

In [16]:
df_map = df.copy()                                    \
           .drop(['India']) \
           .rename(index={
               'Jammu & Kashmir' : 'Jammu and Kashmir disp'
           })

df_map.head(3).fillna('')

,total,male,female,fΔm,urban_total,urban_male,urban_female,urban_fΔm,rural_total,rural_male,rural_female,rural_fΔm,urbanΔrural
,,,,,,,,,,,,,
Delhi,75.8,74.1,77.7,3.6,75.8,74.1,77.8,3.7,74.0,,76.6,,1.8
Kerala,75.0,71.9,78.0,6.1,74.7,71.5,78.0,6.5,75.2,72.3,78.1,5.8,-0.5
Jammu and Kashmir disp,74.3,72.6,76.3,3.7,78.1,76.0,80.5,4.5,72.7,71.1,74.6,3.5,5.4


In [17]:
dd_legend = {
    # '78.0–78.9' : '002000',
    # '77.0–77.9' : '006000',
    # '76.0–76.9' : '009000',
    '75.0–75.9' : '00b800',
    '74.0–74.9' : '00e000',
    '73.0–73.9' : '00ff00',
    '72.0–72.9' : 'b8ff00',
    '71.0–71.9' : 'ffff00',
    '70.0–70.9' : 'ffe000',
    '69.0–69.9' : 'ffc000',
    '68.0–68.9' : 'ffa000',
    '67.0–67.9' : 'ff8000',
    '66.0–66.9' : 'ff5000',
    '65.0–65.9' : 'ff0000',
    # '64.0–64.9' : 'c80000',
    # '63.0–63.9' : '900000',
    # '62.0–62.9' : '400000'
}

def create_legend_code(dd_legend):
    for k, v in dd_legend.items():
        print(f"{{{{Legend|#{v}|{k}}}}}")

create_legend_code(dd_legend)

{{Legend|#00b800|75.0–75.9}}
{{Legend|#00e000|74.0–74.9}}
{{Legend|#00ff00|73.0–73.9}}
{{Legend|#b8ff00|72.0–72.9}}
{{Legend|#ffff00|71.0–71.9}}
{{Legend|#ffe000|70.0–70.9}}
{{Legend|#ffc000|69.0–69.9}}
{{Legend|#ffa000|68.0–68.9}}
{{Legend|#ff8000|67.0–67.9}}
{{Legend|#ff5000|66.0–66.9}}
{{Legend|#ff0000|65.0–65.9}}


In [18]:
def filter_df(df, selected_column):
    filtered_df = df.loc[:, [selected_column]]   \
                    .sort_values(by=selected_column, ascending=False) \
                    .dropna()

    filtered_df['group_label'] = filtered_df[selected_column].map(lambda x: f"{int(x):.1f}–{int(x) + 0.9:.1f}")
    
    
    # filtered_df['group_label'] = filtered_df['group_label'].replace(['79.5–79.99', '79.0–79.49', '78.5–78.99', '78.0–78.49'], '78.0–79.99')
    
    min_value = filtered_df[selected_column].min()
    max_value = filtered_df[selected_column].max()
    
    print(f"           ——— {selected_column} ———")
    print(f"Range: {min_value:.1f} – {max_value:.1f}   " +
          f"({filtered_df[selected_column].idxmin()} – {filtered_df[selected_column].idxmax()})")
    print(f"Number of groups: {filtered_df['group_label'].nunique()}")
    print(f"Number of values: {len(filtered_df)}")

    return filtered_df


df_sel = filter_df(df_map, selected_column='total')
df_sel

           ——— total ———
Range: 65.1 – 75.8   (Chhattisgarh – Delhi)
Number of groups: 9
Number of values: 22


,total,group_label
,,
Delhi,75.8,75.0–75.9
Kerala,75.0,75.0–75.9
Jammu and Kashmir disp,74.3,74.0–74.9
Himachal Pradesh,73.5,73.0–73.9
Tamil Nadu,73.2,73.0–73.9
Maharashtra,72.9,72.0–72.9
Punjab,72.5,72.0–72.9
West Bengal,72.3,72.0–72.9
Andhra Pradesh,70.6,70.0–70.9


In [19]:
def extract_indexes(subdf, dd_legend = dd_legend):
    group_label = subdf['group_label'].iloc[0]
    countries = subdf.index.to_list()
    color = (dd_legend[group_label])
    
    ls_grouping.append(CountryGroup(group_label=group_label, countries=countries, color=color))

    return pd.Series([color, countries], index=['color', 'regions'])


ls_grouping = []
df_grouped = df_sel.groupby(['group_label'])[['group_label']].apply(extract_indexes).loc[::-1]

df_grouped

,color,regions
group_label,,
75.0–75.9,00b800,"[Delhi, Kerala]"
74.0–74.9,00e000,[Jammu and Kashmir disp]
73.0–73.9,00ff00,"[Himachal Pradesh, Tamil Nadu]"
72.0–72.9,b8ff00,"[Maharashtra, Punjab, West Bengal]"
70.0–70.9,ffe000,"[Andhra Pradesh, Uttarakhand, Gujarat, Odisha,..."
69.0–69.9,ffc000,"[Haryana, Karnataka, Jharkhand, Bihar, Rajasthan]"
67.0–67.9,ff8000,"[Assam, Madhya Pradesh]"
66.0–66.9,ff5000,[Uttar Pradesh]
65.0–65.9,ff0000,[Chhattisgarh]


In [20]:
# add record for n/a states
states_na = ['Andaman and Nicobar Islands', 'Arunachal Pradesh', 'Chandigarh', 'Dadra and Nagar Haveli and Daman and Diu', 'Goa',
             'Ladakh disp', 'Lakshadweep', 'Manipur', 'Meghalaya', 'Mizoram', 'Nagaland', 'Puducherry', 'Sikkim', 'Tripura']

ls_on_map = df_map.index.to_list()

assert not [state for state in states_na if state in ls_on_map], "some states, designed to be assign as N/A, are really have values"

ls_grouping.insert(0, CountryGroup(group_label='n/a', countries=states_na, color='e0e0e0'))

In [21]:
def create_map_code_regions(ls_grouping, title=''):
    st = '{"groups":{'
    for group_label, color, regions in ls_grouping[::-1]:
        st_ls_regions = '"' + '","'.join(regions) + '"'
        st_ls_regions = st_ls_regions.replace(' ', '_')
        st += f'"#{color}":{{"label":"{group_label}","paths":[{st_ls_regions}]}},'

    st = st[:-1] + '},"title":"' + title + \
         '","hidden":[],"background":"#fff","borders":"#000","legendFont":"Century Gothic","legendFontColor":"#000","legendBgColor":"#00000000","legendBoxShape":"square","legendBorderColor":"#00000000","legendWidth":484.03571428571445,"areBordersShown":true,"defaultColor":"#d1dbdd","labelsColor":"#6a0707","labelsFont":"Arial","strokeWidth":"medium","areLabelsShown":false,"uncoloredScriptColor":"#ffff33","v5":true,"legendPosition":"custom","legendX":23.82142857142844,"legendY":23.678571428571388,"canvasWidth":1920,"canvasHeight":2280,"legendSize":"large","legendStatus":"show","scalingPatterns":true,"legendRowsSameColor":true,"legendColumnCount":1}'
    
    return st


st = create_map_code_regions(ls_grouping, title='2016 – 2020 (source: SRS)')
print(st)

{"groups":{"#00b800":{"label":"75.0–75.9","paths":["Delhi","Kerala"]},"#00e000":{"label":"74.0–74.9","paths":["Jammu_and_Kashmir_disp"]},"#00ff00":{"label":"73.0–73.9","paths":["Himachal_Pradesh","Tamil_Nadu"]},"#b8ff00":{"label":"72.0–72.9","paths":["Maharashtra","Punjab","West_Bengal"]},"#ffe000":{"label":"70.0–70.9","paths":["Andhra_Pradesh","Uttarakhand","Gujarat","Odisha","Telangana"]},"#ffc000":{"label":"69.0–69.9","paths":["Haryana","Karnataka","Jharkhand","Bihar","Rajasthan"]},"#ff8000":{"label":"67.0–67.9","paths":["Assam","Madhya_Pradesh"]},"#ff5000":{"label":"66.0–66.9","paths":["Uttar_Pradesh"]},"#ff0000":{"label":"65.0–65.9","paths":["Chhattisgarh"]},"#e0e0e0":{"label":"n/a","paths":["Andaman_and_Nicobar_Islands","Arunachal_Pradesh","Chandigarh","Dadra_and_Nagar_Haveli_and_Daman_and_Diu","Goa","Ladakh_disp","Lakshadweep","Manipur","Meghalaya","Mizoram","Nagaland","Puducherry","Sikkim","Tripura"]}},"title":"2016 – 2020 (source: SRS)","hidden":[],"background":"#fff","borders